# BUTD Image Captioning — Training Notebook

End-to-end training notebook for the **Bottom-Up Top-Down (BUTD)** attention model
(Anderson et al., CVPR 2018) on Flickr30k.

### Pipeline overview
1. **Feature extraction** (one-time, ~30 min CPU / ~5 min GPU) — Faster R-CNN extracts 36 region features per image and saves them to `data/butd_features/`.
2. **Training** — A two-layer LSTM decoder with additive attention is trained on the pre-extracted features using teacher forcing.
3. **Evaluation** — BLEU-1/2/3/4 and METEOR are computed on the validation split.
4. **Sample predictions** — Qualitative look at generated vs. reference captions.

Run all cells top-to-bottom. Re-running the notebook after the first full run will skip already-extracted images and reload the best checkpoint automatically.

## 1. Setup

In [1]:
import os
import sys

# Always run from the project root so relative paths work correctly.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working directory:", os.getcwd())

# Make src/ importable without installing the package.
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

Working directory: c:\Users\tochi\OneDrive\Documents\georgia_tech_omscs\deeplearning\final_project\deeplearning-project


In [2]:
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim

from src.data.splits import load_split_file, load_caption_map
from src.data.vocab import Vocabulary
from src.data.butd_dataset import create_butd_dataloader
from src.models.butd_model import BUTDCaptionModel
from src.training.train_butd import train_one_epoch, evaluate_loss, generate_captions
from src.eval.metrics import evaluate_captions

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch version : {torch.__version__}")
print(f"Compute device  : {device}")

PyTorch version : 2.11.0+cpu
Compute device  : cpu


## 2. Configuration

All paths and hyperparameters live here — edit this cell to experiment.

In [4]:
# --- Paths ---
CAPTIONS_PATH  = Path("data/captions.txt")
IMAGES_DIR     = Path("data/Images")
FEATURES_DIR   = Path("data/butd_features")   # output of extract_butd_features.py
VOCAB_PATH     = Path("metadata/vocab/flickr30k_vocab.json")
SPLITS_DIR     = Path("metadata/splits")
CHECKPOINT_DIR = Path("checkpoints/butd")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# --- Model hyperparameters ---
NUM_REGIONS   = 36    # regions extracted per image
FEATURE_DIM   = 1024  # Faster R-CNN TwoMLPHead output dim
EMBED_DIM     = 512
HIDDEN_DIM    = 512
ATTENTION_DIM = 512
DROPOUT       = 0.5

# --- Training hyperparameters ---
BATCH_SIZE  = 32
LR          = 4e-4
GRAD_CLIP   = 5.0
NUM_EPOCHS  = 20
LR_STEP     = 5     # decay LR every this many epochs
LR_GAMMA    = 0.5   # LR decay factor

print("Configuration loaded.")

Configuration loaded.


## 3. Feature Extraction (one-time)

This step runs **once**. If `data/butd_features/` already contains all `.pt` files, it is skipped instantly.

The extraction script uses a pretrained Faster R-CNN (ResNet-50 + FPN, trained on COCO) to produce 36 region features (1024-dim) per image.

**Estimated time**: ~5 min on GPU, ~30–60 min on CPU for the full Flickr30k dataset (~31 k images).

In [5]:
train_ids = load_split_file(SPLITS_DIR / "train.txt")
val_ids   = load_split_file(SPLITS_DIR / "val.txt")
test_ids  = load_split_file(SPLITS_DIR / "test.txt")
all_ids   = train_ids + val_ids + test_ids

FEATURES_DIR.mkdir(parents=True, exist_ok=True)
missing = [iid for iid in all_ids if not (FEATURES_DIR / f"{iid}.pt").exists()]
print(f"Images total   : {len(all_ids)}")
print(f"Features ready : {len(all_ids) - len(missing)}")
print(f"Missing        : {len(missing)}")

Images total   : 31783
Features ready : 0
Missing        : 31783


In [ ]:
if missing:
    print("Running feature extraction …")
    import subprocess
    result = subprocess.run(
        [
            sys.executable, "scripts/extract_butd_features.py",
            "--images-dir",  str(IMAGES_DIR),
            "--output-dir",  str(FEATURES_DIR),
            "--splits-dir",  str(SPLITS_DIR),
            "--num-regions", str(NUM_REGIONS),
            "--batch-size",  "8",
        ],
        capture_output=False,  # stream output to notebook
    )
    if result.returncode != 0:
        raise RuntimeError("Feature extraction failed — check the output above.")
    print("Extraction complete.")
else:
    print("All features already extracted — skipping.")

Running feature extraction …


## 4. Dataset and DataLoaders

In [ ]:
vocab = Vocabulary.load(VOCAB_PATH)
vocab_size = len(vocab.word2idx)
print(f"Vocabulary size : {vocab_size}")

caption_map = load_caption_map(CAPTIONS_PATH)

train_loader = create_butd_dataloader(
    caption_map=caption_map,
    features_dir=FEATURES_DIR,
    split_image_ids=train_ids,
    vocab=vocab,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_loader = create_butd_dataloader(
    caption_map=caption_map,
    features_dir=FEATURES_DIR,
    split_image_ids=val_ids,
    vocab=vocab,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print(f"Train batches : {len(train_loader)}  ({len(train_loader.dataset)} samples)")
print(f"Val batches   : {len(val_loader)}  ({len(val_loader.dataset)} samples)")

In [ ]:
# Sanity-check one batch to verify shapes
sample_batch = next(iter(train_loader))
print("features     :", sample_batch["features"].shape)    # (B, 36, 1024)
print("caption_ids  :", sample_batch["caption_ids"].shape)  # (B, T)
print("target_ids   :", sample_batch["target_ids"].shape)   # (B, T)
print("padding_mask :", sample_batch["padding_mask"].shape) # (B, T)
print("Sample caption:", sample_batch["caption_texts"][0])

## 5. Model

In [ ]:
model = BUTDCaptionModel(
    vocab_size=vocab_size,
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    feature_dim=FEATURE_DIM,
    attention_dim=ATTENTION_DIM,
    dropout=DROPOUT,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")
print(model)

In [ ]:
# Quick forward-pass smoke test
model.eval()
with torch.no_grad():
    test_feats = sample_batch["features"][:2].to(device)
    test_caps  = sample_batch["caption_ids"][:2].to(device)
    logits = model(test_feats, test_caps)
    print(f"Logits shape: {logits.shape}  — expected (2, T, {vocab_size})")
    generated = model.generate(test_feats, vocab.word2idx["<start>"], vocab.word2idx["<end>"])
    print(f"Generated shape: {generated.shape}")
    print("Sample decode:", " ".join(vocab.decode(generated[0].tolist())))
model.train();

## 6. Training

The best checkpoint (lowest validation loss) is saved to `checkpoints/butd/best.pt`.

If a checkpoint already exists from a previous run, re-running this cell will resume training from scratch (the checkpoint is overwritten only when val loss improves). To continue from a saved checkpoint instead, load it in the cell after the loop.

In [ ]:
# CrossEntropyLoss with ignore_index=0 automatically masks <pad> tokens
# (PAD_TOKEN is always index 0 in this vocabulary).
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=LR_STEP, gamma=LR_GAMMA)

train_losses = []
val_losses   = []
best_val_loss = float("inf")

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device, GRAD_CLIP)
    val_loss   = evaluate_loss(model, val_loader, criterion, device)
    scheduler.step()

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    checkpoint_flag = ""
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss,
                "vocab_size": vocab_size,
                "hparams": {
                    "embed_dim": EMBED_DIM,
                    "hidden_dim": HIDDEN_DIM,
                    "feature_dim": FEATURE_DIM,
                    "attention_dim": ATTENTION_DIM,
                },
            },
            CHECKPOINT_DIR / "best.pt",
        )
        checkpoint_flag = "  ← best"

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS}  "
        f"train_loss={train_loss:.4f}  "
        f"val_loss={val_loss:.4f}{checkpoint_flag}"
    )

## 7. Loss Curves

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
epochs = range(1, len(train_losses) + 1)
ax.plot(epochs, train_losses, label="Train", marker="o", markersize=4)
ax.plot(epochs, val_losses,   label="Val",   marker="s", markersize=4)
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-Entropy Loss")
ax.set_title("BUTD Training — Loss Curves")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig("notebooks/butd_loss_curve.png", dpi=120)
plt.show()
print(f"Best val loss: {best_val_loss:.4f}")

## 8. Evaluation (BLEU / METEOR)

Reload the best checkpoint, generate one caption per validation image via greedy decoding, then compute corpus-level scores.

In [ ]:
# Load the best checkpoint
checkpoint_path = CHECKPOINT_DIR / "best.pt"
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
model.load_state_dict(checkpoint["model_state_dict"])
print(f"Loaded checkpoint from epoch {checkpoint['epoch']}  (val_loss={checkpoint['val_loss']:.4f})")

In [ ]:
print("Generating captions on the validation split …")
val_predictions = generate_captions(model, val_loader, vocab, device)
print(f"Generated captions for {len(val_predictions)} unique images.")

In [ ]:
# Build reference dict (only images that were predicted)
val_references = {
    iid: caption_map[iid]
    for iid in val_predictions
    if iid in caption_map
}

scores = evaluate_captions(val_references, val_predictions, require_exact_match=False)

print("\n=== Validation Scores ===")
for metric, score in scores.items():
    print(f"  {metric.upper():8s}: {score:.4f}")

## 9. Sample Predictions

Qualitative check: show the generated caption alongside the five reference captions for a handful of validation images.

In [ ]:
import random
from PIL import Image
import matplotlib.pyplot as plt

sample_ids = random.sample(sorted(val_predictions.keys()), min(6, len(val_predictions)))

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for ax, iid in zip(axes, sample_ids):
    img_path = IMAGES_DIR / iid
    if img_path.exists():
        ax.imshow(Image.open(img_path).convert("RGB"))
    ax.axis("off")
    generated = val_predictions[iid]
    ax.set_title(f"BUTD: {generated}", fontsize=7, wrap=True)

fig.suptitle("BUTD Generated Captions (val split)", fontsize=11)
fig.tight_layout()
fig.savefig("notebooks/butd_sample_predictions.png", dpi=120)
plt.show()

In [ ]:
# Detailed text comparison for the same images
for iid in sample_ids:
    print(f"Image : {iid}")
    print(f"  Generated : {val_predictions[iid]}")
    print("  References:")
    for ref in caption_map.get(iid, []):
        print(f"    - {ref}")
    print()

## 10. Save Predictions for Comparison

Saves predictions in the same JSON format used by `scripts/evaluate_captions.py`, making it easy to compare BUTD against the transformer model later.

In [ ]:
import json

predictions_path = Path("configs/butd_val_predictions.json")
with predictions_path.open("w", encoding="utf-8") as fh:
    json.dump(val_predictions, fh, indent=2, ensure_ascii=True)
print(f"Saved {len(val_predictions)} predictions to {predictions_path}")